# A rule you can say out loud

The holdout is defined now and used later. Take the MD5 digest of `customerID` as a hexadecimal integer. The customer is held out when that integer is divisible by 5. Otherwise the customer is in the fit set. The label is not an input to the split.

That puts 5626 customers in the fit set (1504 leavers) and reserves 1417. This notebook scores rules only on the 5626.

The menu is eight sentences:

1. Month-to-month.
2. Fiber optic.
3. Electronic check.
4. Tenure at most 12.
5. Month-to-month and fiber optic.
6. Month-to-month and tenure at most 12.
7. Online security equal to `No`.
8. Month-to-month, and fiber optic or an electronic check.

For a rule, let $tp$ be flagged leavers, $fp$ flagged stayers, and $fn$ missed leavers. Then

$$
\mathrm{precision} = \dfrac{tp}{tp+fp},
\qquad
\mathrm{recall} = \dfrac{tp}{tp+fn},
\qquad
F_1 = \dfrac{2 \cdot \mathrm{precision} \cdot \mathrm{recall}}{\mathrm{precision}+\mathrm{recall}}.
$$

The winner on the fit rows is sentence 8:

$$
tp = 1123,\quad fp = 1052,\quad fn = 381,\quad tn = 3070.
$$

$$
\mathrm{precision} = \dfrac{1123}{2175},
\qquad
\mathrm{recall} = \dfrac{1123}{1504},
\qquad
F_1 = \dfrac{2246}{3679}.
$$

Check $1123 + 1052 = 2175$ and $1123 + 381 = 1504$. This $F_1$ is larger than the $F_1$ of each other sentence on the same 5626 rows. The 1417 held-out rows are not used to pick the winner.


In [1]:
# Locate the telco table beside this notebook, or under the course root.
import csv
from pathlib import Path

def data_path():
    here = Path.cwd()
    name = "Telco-Customer-Churn.csv"
    for folder in [here, *here.parents]:
        nested = folder / "17-Customer Churn" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists():
            return direct
    raise FileNotFoundError(name)

def read_customers():
    with data_path().open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))
# Eight readable rules. Keep the one with the largest fit-set F1.
import hashlib
from fractions import Fraction

rows = read_customers()

def held(customer_id):
    digest = hashlib.md5(customer_id.encode("utf-8")).hexdigest()
    return int(digest, 16) % 5 == 0

def counts(rule, people):
    tp = fp = fn = tn = 0
    for row in people:
        predicted = rule(row)
        actual = row["Churn"] == "Yes"
        if predicted and actual:
            tp += 1
        elif predicted and not actual:
            fp += 1
        elif actual:
            fn += 1
        else:
            tn += 1
    return tp, fp, fn, tn

def f1_of(tp, fp, fn):
    precision = Fraction(tp, tp + fp)
    recall = Fraction(tp, tp + fn)
    return 2 * precision * recall / (precision + recall)

fit = [row for row in rows if not held(row["customerID"])]
test = [row for row in rows if held(row["customerID"])]
assert len(fit) == 5626 and len(test) == 1417
assert sum(row["Churn"] == "Yes" for row in fit) == 1504
menu = {
    "month": lambda row: row["Contract"] == "Month-to-month",
    "fiber": lambda row: row["InternetService"] == "Fiber optic",
    "check": lambda row: row["PaymentMethod"] == "Electronic check",
    "tenure": lambda row: int(row["tenure"]) <= 12,
    "month fiber": lambda row: (
        row["Contract"] == "Month-to-month" and row["InternetService"] == "Fiber optic"
    ),
    "month tenure": lambda row: (
        row["Contract"] == "Month-to-month" and int(row["tenure"]) <= 12
    ),
    "no security": lambda row: row["OnlineSecurity"] == "No",
    "month and fiber or check": lambda row: (
        row["Contract"] == "Month-to-month"
        and (
            row["InternetService"] == "Fiber optic"
            or row["PaymentMethod"] == "Electronic check"
        )
    ),
}
scores = {}
for name, rule in menu.items():
    tp, fp, fn, tn = counts(rule, fit)
    scores[name] = f1_of(tp, fp, fn)
    print(name, scores[name])
winner = max(scores, key=scores.get)
print(winner)
assert winner == "month and fiber or check"
assert scores[winner] == Fraction(2246, 3679)
assert all(scores[winner] > scores[name] for name in scores if name != winner)
tp, fp, fn, tn = counts(menu[winner], fit)
assert (tp, fp, fn, tn) == (1123, 1052, 381, 3070)
assert Fraction(tp, tp + fp) == Fraction(1123, 2175)
assert Fraction(tp, tp + fn) == Fraction(1123, 1504)


month 1348/2307
fiber 422/799
check 70/137
tenure 168/325
month fiber 951/1615
month tenure 833/1551
no security 2350/4319
month and fiber or check 2246/3679
month and fiber or check


## Pitfall

The menu is these eight sentences. A column that was left off the menu stays off after the holdout is scored. Adding it later would be a second look at the same contest.

## Summary

The fit set has 5626 customers. The winning sentence is: month-to-month, and fiber optic or an electronic check. On the fit rows its confusion counts are 1123, 1052, 381, and 3070, and its $F_1$ is $2246/3679$.
